# Impacts of Borderline Cleaning

### Classification without RUS and SMOTE - (SVM, and GRU)

In [16]:
import pickle
import numpy as np

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

# ── Load hybrid_72 only ──
BASE = "./final_split_data_HybridNorm_timesteps"
datasets = {}
folder = f"{BASE}/hybrid_72"
X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")
datasets["hybrid_72"] = {
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
}

# ── Summary ───────────────────────────────────────────────────
print(f"{'Dataset':<15} {'Train shape':<25} {'Val shape':<22} {'Test shape':<22} {'Class 0':>8} {'Class 1':>8} {'Ratio':>8}")
print("─" * 115)
for name, d in datasets.items():
    counts = np.bincount(d["y_train"].astype(int))
    ratio  = counts[0] / counts[1]
    print(f"{name:<15} {str(d['X_train'].shape):<25} {str(d['X_val'].shape):<22} {str(d['X_test'].shape):<22} "
          f"{counts[0]:>8} {counts[1]:>8} {ratio:>7.1f}x")

Dataset         Train shape               Val shape              Test shape              Class 0  Class 1    Ratio
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
hybrid_72       (12455, 72, 10)           (1780, 72, 10)         (3559, 72, 10)            12337      118   104.6x


### Vectorization for SVM

In [4]:
# ══════════════════════════════════════════════════════════════
# CATCH22 EXTRACTION — hybrid_72 only
# ══════════════════════════════════════════════════════════════
import os
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22

SAVE_DIR = "./catch22_features_timesteps"
os.makedirs(SAVE_DIR, exist_ok=True)


def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 500) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️  {n_bad} non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]
            col[~np.isfinite(col)] = np.nanmedian(col)
            X[:, :, c] = col

    X_sktime = X.transpose(0, 2, 1).astype(np.float64)
    chunks   = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]
    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk) for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels
    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected catch22 output shape: {X_feat.shape}  "
        f"(expected ({n_samples}, {expected_cols}))"
    )
    return X_feat


# ── Sanity check ───────────────────────────────────────────────
d = datasets["hybrid_72"]
X = d["X_train"]
print(f"Pre-extraction check — hybrid_72:")
print(f"  Finite: {np.isfinite(X).all()}  Min: {X.min():.4f}  Max: {X.max():.4f}  NaNs: {(~np.isfinite(X)).sum()}")

# ── Extraction ─────────────────────────────────────────────────
print("\nExtracting catch22 features for hybrid_72 …")

X_tr = extract_catch22(d["X_train"], label="hybrid_72/train", n_jobs=-1)
X_va = extract_catch22(d["X_val"],   label="hybrid_72/val",   n_jobs=-1)
X_te = extract_catch22(d["X_test"],  label="hybrid_72/test",  n_jobs=-1)

np.savez(os.path.join(SAVE_DIR, "hybrid_72_train.npz"), X=X_tr, y=d["y_train"])
np.savez(os.path.join(SAVE_DIR, "hybrid_72_val.npz"),   X=X_va, y=d["y_val"])
np.savez(os.path.join(SAVE_DIR, "hybrid_72_test.npz"),  X=X_te, y=d["y_test"])

print(f"  ✓  train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}  → {SAVE_DIR}/")

Pre-extraction check — hybrid_72:
  Finite: True  Min: -6.5219  Max: 93.5328  NaNs: 0

Extracting catch22 features for hybrid_72 …
    hybrid_72/train — 12455 samples in 25 chunks …
    hybrid_72/val — 1780 samples in 4 chunks …
    hybrid_72/test — 3559 samples in 8 chunks …
  ✓  train (12455, 220)  val (1780, 220)  test (3559, 220)  → ./catch22_features_timesteps/


### Missing value imputation from Catch22 - Loading the data

In [5]:
# ══════════════════════════════════════════════════════════════
# IMPUTE — hybrid_72 only
# ══════════════════════════════════════════════════════════════
from sklearn.impute import SimpleImputer
import os
import numpy as np

SAVE_DIR = "./catch22_features_timesteps"

paths = {
    "train": os.path.join(SAVE_DIR, "hybrid_72_train.npz"),
    "val":   os.path.join(SAVE_DIR, "hybrid_72_val.npz"),
    "test":  os.path.join(SAVE_DIR, "hybrid_72_test.npz"),
}

tr = np.load(paths["train"])
va = np.load(paths["val"])
te = np.load(paths["test"])

X_tr, y_tr = tr["X"], tr["y"]
X_va, y_va = va["X"], va["y"]
X_te, y_te = te["X"], te["y"]

n_bad_tr = (~np.isfinite(X_tr)).sum()
n_bad_va = (~np.isfinite(X_va)).sum()
n_bad_te = (~np.isfinite(X_te)).sum()

if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:
    print(f"  ⚠️  [hybrid_72] non-finite values — "
          f"train: {n_bad_tr}  val: {n_bad_va}  test: {n_bad_te}  → imputing")

    X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
    X_va = np.where(np.isfinite(X_va), X_va, np.nan)
    X_te = np.where(np.isfinite(X_te), X_te, np.nan)

    imp  = SimpleImputer(strategy="median")
    X_tr = imp.fit_transform(X_tr)
    X_va = imp.transform(X_va)
    X_te = imp.transform(X_te)

    np.savez(paths["train"], X=X_tr, y=y_tr)
    np.savez(paths["val"],   X=X_va, y=y_va)
    np.savez(paths["test"],  X=X_te, y=y_te)
    print(f"  ✓  [hybrid_72] cleaned and saved")
else:
    print(f"  ✓  [hybrid_72] no NaN or Inf — files unchanged")

datasets_svm = {
    "hybrid_72": {
        "X_train": X_tr, "y_train": y_tr,
        "X_val":   X_va, "y_val":   y_va,
        "X_test":  X_te, "y_test":  y_te,
    }
}

print(f"\n✅  datasets_svm loaded: {list(datasets_svm.keys())}")
print(f"    hybrid_72  train={X_tr.shape}  val={X_va.shape}  test={X_te.shape}")

  ⚠️  [hybrid_72] non-finite values — train: 225421  val: 36505  test: 71239  → imputing
  ✓  [hybrid_72] cleaned and saved

✅  datasets_svm loaded: ['hybrid_72']
    hybrid_72  train=(12455, 220)  val=(1780, 220)  test=(3559, 220)


### TomekLinks - EditedNearestNeighbours - NearMiss 

In [6]:
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours, NearMiss
import numpy as np

# base data from datasets_svm
X_tr_base = datasets_svm["hybrid_72"]["X_train"]
y_tr_base = datasets_svm["hybrid_72"]["y_train"].astype(int)
X_va      = datasets_svm["hybrid_72"]["X_val"]
y_va      = datasets_svm["hybrid_72"]["y_val"]
X_te      = datasets_svm["hybrid_72"]["X_test"]
y_te      = datasets_svm["hybrid_72"]["y_test"]

print(f"Original train: {X_tr_base.shape}  NSEP={(y_tr_base==0).sum()}  SEP={(y_tr_base==1).sum()}")

# ── Apply three techniques ─────────────────────────────────────
samplers = {
    "tomek":    TomekLinks(),
    "enn":      EditedNearestNeighbours(n_neighbors=3),
    "nearmiss": NearMiss(version=3),
}

datasets_svm_cleaned = {}

for method_name, sampler in samplers.items():
    print(f"\n── {method_name.upper()} ──")
    X_clean, y_clean = sampler.fit_resample(X_tr_base, y_tr_base)
    print(f"  After : {X_clean.shape}  NSEP={(y_clean==0).sum()}  SEP={(y_clean==1).sum()}")
    print(f"  Removed {X_tr_base.shape[0] - X_clean.shape[0]} samples")

    datasets_svm_cleaned[f"hybrid_72_{method_name}"] = {
        "X_train": X_clean, "y_train": y_clean,
        "X_val":   X_va,    "y_val":   y_va,
        "X_test":  X_te,    "y_test":  y_te,
    }

print(f"\n✅  Cleaned datasets in memory: {list(datasets_svm_cleaned.keys())}")

Original train: (12455, 220)  NSEP=12337  SEP=118

── TOMEK ──
  After : (12436, 220)  NSEP=12318  SEP=118
  Removed 19 samples

── ENN ──
  After : (12253, 220)  NSEP=12135  SEP=118
  Removed 202 samples

── NEARMISS ──
  After : (236, 220)  NSEP=118  SEP=118
  Removed 12219 samples

✅  Cleaned datasets in memory: ['hybrid_72_tomek', 'hybrid_72_enn', 'hybrid_72_nearmiss']


### Classification

In [5]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [7]:
from sklearn.svm import LinearSVC

best_model_class = LinearSVC
best_params      = {"C": 1, "class_weight": "balanced", "max_iter": 3000}

DATASET_NAMES = {
    "hybrid_72_tomek"   : "hybrid_72_tomek",
    "hybrid_72_enn"     : "hybrid_72_enn",
    "hybrid_72_nearmiss": "hybrid_72_nearmiss",
}

print(f"{'═'*60}")
print(f"  Classifier : SVM")
print(f"  Model      : LinearSVC  C=1  cw=balanced")
print(f"{'═'*60}")

for norm_key, norm_label in DATASET_NAMES.items():
    if norm_key not in datasets_svm_cleaned:
        print(f"  [{norm_key}] not found — skipping")
        continue

    d = datasets_svm_cleaned[norm_key]
    print(f"\n  ── Dataset: {norm_label} ──")

    metrics_list = []
    for run in range(2):
        model   = best_model_class(**best_params)
        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.2f}s")

    filepath = os.path.join(RESULTS_DIR, f"svm_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {norm_label}")

print(f"\n\n✅  All SVM experiments complete.")
print(f"    Results saved to : {os.path.abspath(RESULTS_DIR)}/")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

════════════════════════════════════════════════════════════
  Classifier : SVM
  Model      : LinearSVC  C=1  cw=balanced
════════════════════════════════════════════════════════════

  ── Dataset: hybrid_72_tomek ──
    Run 1 — TSS=0.3690  F1=0.0784  Train=245.01s
    Run 2 — TSS=0.3690  F1=0.0784  Train=235.38s

───────────────────────────────────────────────────────
  SVM | hybrid_72_tomek
───────────────────────────────────────────────────────
  Run 1: TP=16  TN=3167  FP=358  FN=18
         TSS=0.3690  HSS1=0.0620  HSS2=0.0620  GSS=0.0320
         Recall=0.4706  F1=0.0784  Acc=0.8944
         Train=245.01s  Infer=0.0007s

  Run 2: TP=16  TN=3167  FP=358  FN=18
         TSS=0.3690  HSS1=0.0620  HSS2=0.0620  GSS=0.0320
         Recall=0.4706  F1=0.0784  Acc=0.8944
         Train=235.38s  Infer=0.0035s

  ── Average of 2 runs ──
  tss          : 0.3690
  hss1         : 0.0620
  hss2         : 0.0620
  gss          : 0.0320
  recall       : 0.4706
  f1           : 0.0784
  accuracy   

/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    Run 1 — TSS=0.3973  F1=0.0823  Train=239.17s


/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    Run 2 — TSS=0.3973  F1=0.0823  Train=234.41s

───────────────────────────────────────────────────────
  SVM | hybrid_72_enn
───────────────────────────────────────────────────────
  Run 1: TP=17  TN=3163  FP=362  FN=17
         TSS=0.3973  HSS1=0.0659  HSS2=0.0659  GSS=0.0341
         Recall=0.5000  F1=0.0823  Acc=0.8935
         Train=239.17s  Infer=0.0107s

  Run 2: TP=17  TN=3163  FP=362  FN=17
         TSS=0.3973  HSS1=0.0659  HSS2=0.0659  GSS=0.0341
         Recall=0.5000  F1=0.0823  Acc=0.8935
         Train=234.41s  Infer=0.0007s

  ── Average of 2 runs ──
  tss          : 0.3973
  hss1         : 0.0659
  hss2         : 0.0659
  gss          : 0.0341
  recall       : 0.5000
  f1           : 0.0823
  accuracy     : 0.8935
  train_time   : 236.7926
  infer_time   : 0.0057
───────────────────────────────────────────────────────

  ── Dataset: hybrid_72_nearmiss ──
    Run 1 — TSS=0.2231  F1=0.0260  Train=1.45s
    Run 2 — TSS=0.2231  F1=0.0260  Train=1.44s

────────────────────

In [17]:
# ══════════════════════════════════════════════════════════════
# UNDERSAMPLING — hybrid_72
# ══════════════════════════════════════════════════════════════
from imblearn.under_sampling import TomekLinks, EditedNearestNeighbours, NearMiss

d        = datasets["hybrid_72"]
n, t, c  = d["X_train"].shape
X_flat   = d["X_train"].reshape(n, -1)
y_int    = d["y_train"].astype(int)

samplers = {
    "tomek"   : TomekLinks(),
    "enn"     : EditedNearestNeighbours(n_neighbors=3),
    "nearmiss": NearMiss(version=3),
}

datasets_gru_cleaned = {}
for method_name, sampler in samplers.items():
    print(f"\n── {method_name.upper()} ──")
    X_clean_flat, y_clean = sampler.fit_resample(X_flat, y_int)
    X_clean = X_clean_flat.reshape(-1, t, c)
    print(f"  After : {X_clean.shape}  NSEP={(y_clean==0).sum()}  SEP={(y_clean==1).sum()}")
    print(f"  Removed {n - X_clean.shape[0]} samples")
    datasets_gru_cleaned[f"hybrid_72_{method_name}"] = {
        "X_train": X_clean,          "y_train": y_clean.astype(np.float32),
        "X_val":   d["X_val"],       "y_val":   d["y_val"],
        "X_test":  d["X_test"],      "y_test":  d["y_test"],
    }

print(f"\n✅  Cleaned: {list(datasets_gru_cleaned.keys())}")


── TOMEK ──
  After : (12442, 72, 10)  NSEP=12324  SEP=118
  Removed 13 samples

── ENN ──
  After : (12292, 72, 10)  NSEP=12174  SEP=118
  Removed 163 samples

── NEARMISS ──
  After : (236, 72, 10)  NSEP=118  SEP=118
  Removed 12219 samples

✅  Cleaned: ['hybrid_72_tomek', 'hybrid_72_enn', 'hybrid_72_nearmiss']


### GRU 

In [18]:
import numpy as np
from sklearn.metrics import confusion_matrix
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    tss       = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected  = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1      = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom     = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2      = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss         = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }

def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")

def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

In [20]:
# ══════════════════════════════════════════════════════════════
# GRU — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(30):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


# ══════════════════════════════════════════════════════════════
# RUN EXPERIMENTS
# ══════════════════════════════════════════════════════════════
best_params = {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001, "batch_size": 32}

print(f"{'═'*60}")
print(f"  Classifier : GRU (PyTorch)")
print(f"  Best params: {best_params}")
print(f"{'═'*60}")

for norm_key, d in datasets_gru_cleaned.items():
    cr = int((d["y_train"] == 0).sum() / max((d["y_train"] == 1).sum(), 1))
    print(f"\n  ── Dataset: {norm_key}  (class ratio {cr}:1) ──")
    print(f"     Train: NSEP={(d['y_train']==0).sum():.0f}  SEP={(d['y_train']==1).sum():.0f}")

    metrics_list = []
    for run in range(2):
        print(f"    Run {run+1} …", flush=True)
        metrics, _ = train_and_evaluate_gru(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio = cr
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.1f}s")

    filepath = os.path.join(RESULTS_DIR, f"gru_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {norm_key}")

print(f"\n✅  All GRU experiments complete.")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

Using device: mps
════════════════════════════════════════════════════════════
  Classifier : GRU (PyTorch)
  Best params: {'units': 128, 'dropout': 0.3, 'layers': 1, 'lr': 0.001, 'batch_size': 32}
════════════════════════════════════════════════════════════

  ── Dataset: hybrid_72_tomek  (class ratio 104:1) ──
     Train: NSEP=12324  SEP=118
    Run 1 …
    Run 1 — TSS=0.5548  F1=0.0947  Train=119.3s
    Run 2 …
    Run 2 — TSS=0.5780  F1=0.1139  Train=88.4s

───────────────────────────────────────────────────────
  GRU | hybrid_72_tomek
───────────────────────────────────────────────────────
  Run 1: TP=23  TN=3096  FP=429  FN=11
         TSS=0.5548  HSS1=0.0783  HSS2=0.0783  GSS=0.0407
         Recall=0.6765  F1=0.0947  Acc=0.8764
         Train=119.30s  Infer=0.0944s

  Run 2: TP=23  TN=3178  FP=347  FN=11
         TSS=0.5780  HSS1=0.0981  HSS2=0.0981  GSS=0.0516
         Recall=0.6765  F1=0.1139  Acc=0.8994
         Train=88.44s  Infer=0.0948s

  ── Average of 2 runs ──
  tss    